# LIGER best - scib_sim1

Seeds: `1-10`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR


In [1]:
import gc
import importlib.util
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics'
os.environ["R_HOME"] = f"{PROJECT_ROOT}/CondaEnv/pyliger_env/lib/R"
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd

OPTIMIZER_PATH = (
    f"{PROJECT_ROOT}/scripts/optuna_multi_metrics/"
    "optuna_liger/optimize_liger_multi_metrics.py"
)
optimizer_spec = importlib.util.spec_from_file_location(
    "liger_opt",
    OPTIMIZER_PATH,
)
if optimizer_spec is None or optimizer_spec.loader is None:
    raise ImportError(f"Cannot load LIGER optimizer: {OPTIMIZER_PATH}")
liger_opt = importlib.util.module_from_spec(optimizer_spec)
sys.modules[optimizer_spec.name] = liger_opt
optimizer_spec.loader.exec_module(liger_opt)


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATASET_NAME = 'scib_sim1'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Batch'
LABEL_KEY = 'Group'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

SEEDS = tuple(range(1, 11))
MODE = "core"
LAYER = "counts"
N_NEIGHBORS = 15
EMBED_KEY = "X_liger"

liger_opt.DATASET_NAME = DATASET_NAME
liger_opt.DATA_PATH = DATA_PATH
liger_opt.BATCH_KEY = BATCH_KEY
liger_opt.LABEL_KEY = LABEL_KEY
liger_opt.MODE = MODE
liger_opt.LAYER = LAYER
liger_opt.USE_BAMI = "BAMI" in SELECTED_METRICS
liger_opt.USE_CASW = "CASW" in SELECTED_METRICS
liger_opt.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
liger_opt.USE_TRAJ = "TRAJ" in SELECTED_METRICS
liger_opt.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
liger_opt.USE_BRAS = "BRAS" in SELECTED_METRICS
liger_opt.USE_KBET = "KBET" in SELECTED_METRICS
liger_opt.USE_CPCR = "CPCR" in SELECTED_METRICS

assert DATASET_NAME == 'scib_sim1'
assert BATCH_KEY == 'Batch'
assert LABEL_KEY == 'Group'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')
assert SEEDS == tuple(range(1, 11))
assert liger_opt.CORE_MAX_ITERS == 30
assert liger_opt.N_NEIGHBORS == N_NEIGHBORS
assert liger_opt.EMBED_KEY == EMBED_KEY

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(SEEDS),
        "mode": MODE,
        "layer": LAYER,
    },
    name="configuration",
)


dataset                                              scib_sim1
data_path    /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                Batch
label_key                                                Group
metrics      [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
mode                                                      core
layer                                                   counts
Name: configuration, dtype: object

In [3]:
liger_opt.validate_metric_dependencies()
liger_opt.validate_configuration(list(SELECTED_METRICS))
liger_opt.validate_dataset_contents()


Dataset preflight: n_obs=12097 n_batches=6 n_labels=7
ISOLISI preflight: min_batch_coverage=2 isolated_labels=['Group7']
CPCR preflight: eligible_labels=7


In [4]:
OPTUNA_STUDY_NAME = 'liger_scib_sim1_joint_expanded_metrics_scgraphm_v1'
OPTUNA_BEST_TRIAL = 74
OPTUNA_BEST_VALUE = 0.8738907224919364
OPTUNA_BEST_PARAMS = {'k': 26, 'value_lambda': 5.474813020467679}

LIGER_BEST_ARGS = {
    "k": OPTUNA_BEST_PARAMS["k"],
    "value_lambda": OPTUNA_BEST_PARAMS["value_lambda"],
    "max_iters": 30,
    "mode": "core",
    "layer": "counts",
}


def run_liger_best(adata, seed):
    return liger_opt.run_liger(
        adata=adata,
        k=LIGER_BEST_ARGS["k"],
        lam=LIGER_BEST_ARGS["value_lambda"],
        max_iterations=LIGER_BEST_ARGS["max_iters"],
        mode=LIGER_BEST_ARGS["mode"],
        layer=LIGER_BEST_ARGS["layer"],
        seed=seed,
    )


pd.Series(
    {
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **LIGER_BEST_ARGS,
    },
    name="best_setting",
)


study           liger_scib_sim1_joint_expanded_metrics_scgraph...
best_trial                                                     74
best_value                                               0.873891
k                                                              26
value_lambda                                             5.474813
max_iters                                                      30
mode                                                         core
layer                                                      counts
Name: best_setting, dtype: object

In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    return liger_opt.compute_joint_metrics(adata_raw, adata_int)


In [6]:
results = []

for seed in SEEDS:
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_liger_best(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 1 =====


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/pyliger_env/lib/python3.10/site-packages/numba/np/ufunc/dufunc.py:344: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the

Removing 28 genes not expressing in Batch1.
Removing 26 genes not expressing in Batch2.


Removing 28 genes not expressing in Batch3.
Removing 24 genes not expressing in Batch4.
Index(['Gene148', 'Gene1364', 'Gene1573', 'Gene1596', 'Gene1989', 'Gene2028',
       'Gene2271', 'Gene2661', 'Gene3065', 'Gene3185', 'Gene3587', 'Gene4753',
       'Gene5428', 'Gene5635', 'Gene6048', 'Gene6098', 'Gene6420', 'Gene7428',
       'Gene8000', 'Gene8427', 'Gene8705', 'Gene8711', 'Gene9481', 'Gene9519'],
      dtype='object', name='gene')


Removing 27 genes not expressing in Batch5.
Removing 27 genes not expressing in Batch6.


...

resolution: 1.8, bAMI: 0.9175648488150469


Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.8950975887427134


optimised clustering against liger_metric_label
optimal cluster resolution: 0.6
optimal score: 0.9739129054407166
BRAS...
seed           10.000000
BAMI            0.973913
CASW            0.668676
SCGRAPHM        0.534908
ISOLISI         1.000000
BRAS            0.954378
KBET            0.801355
CPCR            0.986678
bio_score       0.794374
batch_score     0.914137
joint_score     0.842279
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,1,0.990123,0.677408,0.679922,1.000000,0.958874,0.839650,0.989772,0.836863,0.929432,0.873891
1,2,0.991678,0.637789,0.469387,1.000000,0.957513,0.824589,0.991553,0.774714,0.924552,0.834649
2,3,0.951231,0.645359,0.470487,1.000000,0.925093,0.712429,0.949283,0.766769,0.862268,0.804969
3,4,0.988379,0.631450,0.369704,1.000000,0.960996,0.804379,0.990186,0.747383,0.918520,0.815838
4,5,0.993519,0.658597,0.717419,1.000000,0.964788,0.804383,0.988634,0.842384,0.919268,0.873137
5,6,0.868641,0.633012,0.465446,0.966426,0.875442,0.706913,0.980287,0.733381,0.854214,0.781714
6,7,0.974017,0.659325,0.520579,1.000000,0.943565,0.825628,0.966548,0.788480,0.911913,0.837853
7,8,0.958690,0.640901,0.608211,1.000000,0.932893,0.749928,0.973574,0.801950,0.885465,0.835356
8,9,0.911352,0.639411,0.634002,0.988476,0.899932,0.788694,0.980274,0.793310,0.889633,0.831839
9,10,0.973913,0.668676,0.534908,1.000000,0.954378,0.801355,0.986678,0.794374,0.914137,0.842279


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.960154,0.038757,0.868641,0.993519
CASW,0.649193,0.015009,0.631450,0.677408
SCGRAPHM,0.547006,0.104324,0.369704,0.717419
ISOLISI,0.995490,0.010279,0.966426,1.000000
BRAS,0.937348,0.028130,0.875442,0.964788
KBET,0.785795,0.044525,0.706913,0.839650
CPCR,0.979679,0.012715,0.949283,0.991553
bio_score,0.787961,0.032968,0.733381,0.842384
batch_score,0.900940,0.025209,0.854214,0.929432
joint_score,0.833153,0.026669,0.781714,0.873891
